First we sanity check the corpus...
Let's look at the stories generated and visually ensure they make sense.

In [27]:
import random
import textwrap
import sys

from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

CORPUS_DIR = PROJECT_ROOT / "datasets" / "qwen-emotion-stories" / "corpus"

from core.shards import read_shards
from core.utils import emotion_words_named

In [28]:
from collections import Counter

rows = read_shards(CORPUS_DIR, sample=False)
print(f"{len(rows)} stories")
print(Counter(r["emotion"] for r in rows))

807 stories
Counter({'neutral': 98, 'surprised': 91, 'ashamed': 68, 'disgusted': 65, 'desperate': 64, 'afraid': 59, 'sad': 59, 'calm': 58, 'angry': 54, 'proud': 52, 'excited': 50, 'joyful': 48, 'lonely': 41})


In [32]:
def show_samples(n: int = 2, seed: int | None = None, emotion: str | None = None, sample: bool = True, prompts: bool = True) -> None:
    rows = read_shards(CORPUS_DIR, sample=sample)
    if emotion:
        rows = [r for r in rows if r["emotion"] == emotion]

    for row in random.Random(seed).sample(rows, min(n, len(rows))):
        named = emotion_words_named(row["text"])
        print("=" * 88)
        print(f"{row['emotion']}  |  {row['topic']}  |  #{row['index']}")
        print(f"{len(row['text'].split())} words  |  ends: {row['text'].rstrip()[-1]!r}  "
              f"|  names: {named or 'none'}")
        if prompts:
            print("-" * 88)
            print(textwrap.indent(row["prompt"], "  "))
        print("-" * 88)
        print(textwrap.fill(row["text"], width=88))
        print()

show_samples(2, sample=False)

afraid  |  A chef receives a harsh review from a food critic  |  #1
80 words  |  ends: '.'  |  names: none
----------------------------------------------------------------------------------------
  Write a short story (roughly one paragraph) based on the following premise.

  Topic: A chef receives a harsh review from a food critic

  The story should follow a character who is feeling afraid.

  Write the story in English. Use either third-person or first-person narration.

  Write between 90 and 130 words. Finish the final sentence. Do not write a title.

  The character is ALREADY feeling afraid in the very first sentence. Open inside the scene, at the moment the feeling is strongest. Do not begin with backstory, scene-setting, or a build-up towards the feeling, and do not begin with "Once upon a time".

  ONE STATE ONLY: the character feels afraid and nothing else, from the first word to the last. Nothing in the story relieves, resolves, or complicates the feeling - no turn, no twis

Next up we need to extract activations from all layers to do our difference of means probe

In [26]:
## get activations

from core.models import Model

m = Model()
m.load_weights()
texts = (row["text"] for row in rows)

inputs = m.tok(texts, return_tensors="pt", padding=True).to(m.device)
out = m.model(**inputs, output_hidden_states=True, max_new_tokens=512, do_sample=False, temperature=0.8, top_p=0.9)
print(out.hidden_states.shape)

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 160.34it/s]


ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).